# Enfermedad cardiaca: factores de riesgo y modelos predictivos (CRISP-DM)

**Autor:** Alejandro Pedreros  
**Fecha:** 13/11/2026

Dataset: Heart Disease Health Indicators (BRFSS 2015), Kaggle, licencia CC0.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/esteban-nieto/heart-disease-crispdm.git"

if EN_COLAB:
    if not Path("/content/heart-disease-crispdm").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO,
                        "/content/heart-disease-crispdm"], check=True)
    os.chdir("/content/heart-disease-crispdm")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   check=True)

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
for carpeta in ["data", "reports/figures"]:
    (RAIZ / carpeta).mkdir(parents=True, exist_ok=True)

%matplotlib inline

from IPython.display import display

def mostrar(tabla, titulo=None):
    """Muestra una tabla con formato: encabezado azul, decimales fijos y barras de color en las columnas numericas."""
    if hasattr(tabla, "to_frame"):
        tabla = tabla.to_frame()
    numericas = tabla.select_dtypes("number").columns.tolist()
    estilo = (tabla.style
              .format(precision=4, na_rep="-")
              .set_table_styles([
                  {"selector": "th", "props": [("background-color", "#2a7ab0"), ("color", "white"),
                                                ("font-weight", "bold"), ("padding", "6px 10px")]},
                  {"selector": "td", "props": [("padding", "5px 10px")]},
                  {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "1.1em"),
                                                     ("font-weight", "bold"), ("color", "#2a7ab0"),
                                                     ("text-align", "left"), ("padding-bottom", "6px")]},
              ]))
    if numericas:
        estilo = estilo.background_gradient(cmap="Blues", subset=numericas)
    if titulo:
        estilo = estilo.set_caption(titulo)
    display(estilo)


## Fase 1 (CRISP-DM): comprensión del negocio y de los datos

### Diabetes tiene códigos 0, 1 y 2; se trata como categórica en las pruebas.

CRISP-DM fases 1 y 2: comprensión del negocio y de los datos (EDA).

In [2]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu

DATOS = RAIZ / "data" / "heart_disease_health_indicators_BRFSS2015.csv"
FIGURAS = RAIZ / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATOS)
TARGET = "HeartDiseaseorAttack"
NUMERICAS = ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Age", "Education", "Income"]
BINARIAS = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "Diabetes",
            "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
            "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"]

## Fase 1: comprensión del negocio

In [3]:
print("=== FASE 1: COMPRENSIÓN DEL NEGOCIO ===")
print("Objetivo: identificar factores de salud y hábitos asociados a infarto o "
      "enfermedad cardiaca (HeartDiseaseorAttack) y estimar la probabilidad de "
      "que una persona la presente, para apoyar la prevención.")
print("Fuente: encuesta BRFSS 2015 (CDC, EE. UU.), datos autoreportados.")

=== FASE 1: COMPRENSIÓN DEL NEGOCIO ===
Objetivo: identificar factores de salud y hábitos asociados a infarto o enfermedad cardiaca (HeartDiseaseorAttack) y estimar la probabilidad de que una persona la presente, para apoyar la prevención.
Fuente: encuesta BRFSS 2015 (CDC, EE. UU.), datos autoreportados.


## Fase 2: comprensión de los datos

### Distribuciones

Asociación numéricas vs target Asociación binarias vs target Correlación con el target Tasa de evento por edad (código BRFSS 1-13) y por salud general

In [4]:
print("\n=== FASE 1 (CRISP-DM): COMPRENSIÓN DE LOS DATOS ===")
print("Filas x columnas:", df.shape)
print("Nulos totales:", int(df.isna().sum().sum()))
print("Duplicados:", int(df.duplicated().sum()))
print("\nBalance del target (con duplicados):")
mostrar(df[TARGET].value_counts(normalize=True).round(4))

df_sin_dup = df.drop_duplicates()
print(f"\nFilas tras eliminar duplicados: {len(df_sin_dup)}")
print("Balance del target (sin duplicados):")
mostrar(df_sin_dup[TARGET].value_counts(normalize=True).round(4))

resumen = df_sin_dup[NUMERICAS + BINARIAS].describe().T.round(2)
resumen.to_csv(RAIZ / "reports" / "estadisticas_descriptivas.csv")



filas = []
for col in NUMERICAS:
    a = df_sin_dup.loc[df_sin_dup[TARGET] == 0, col]
    b = df_sin_dup.loc[df_sin_dup[TARGET] == 1, col]
    _, p = mannwhitneyu(a, b)
    filas.append({"variable": col, "media_sin_evento": round(a.mean(), 2),
                  "media_con_evento": round(b.mean(), 2),
                  "diferencia_std": round((b.mean() - a.mean()) / df_sin_dup[col].std(), 4),
                  "p_valor": p})
tabla_num = pd.DataFrame(filas)

filas = []
for col in BINARIAS:
    tabla = pd.crosstab(df_sin_dup[col], df_sin_dup[TARGET])
    _, p, _, _ = chi2_contingency(tabla)
    tasas = df_sin_dup.groupby(col)[TARGET].mean().round(4).to_dict()
    filas.append({"variable": col, "tasa_evento_por_nivel": tasas, "p_valor": p})
tabla_cat = pd.DataFrame(filas)

tabla_num.to_csv(RAIZ / "reports" / "asociacion_numericas_con_riesgo.csv", index=False)
tabla_cat.to_csv(RAIZ / "reports" / "asociacion_binarias_con_riesgo.csv", index=False)
print("\nAsociación numéricas vs target:")
mostrar(tabla_num)
print("\nAsociación binarias vs target:")
mostrar(tabla_cat)

corr = df_sin_dup[NUMERICAS + BINARIAS + [TARGET]].corr()[TARGET].drop(TARGET)
corr.sort_values().to_csv(RAIZ / "reports" / "correlacion_con_target.csv", header=["r"])





print("\nFiguras guardadas en reports/figures/")



=== FASE 1 (CRISP-DM): COMPRENSIÓN DE LOS DATOS ===
Filas x columnas: (253680, 22)
Nulos totales: 0


Duplicados: 23899

Balance del target (con duplicados):
HeartDiseaseorAttack
0.0    0.9058
1.0    0.0942



Filas tras eliminar duplicados: 229781
Balance del target (sin duplicados):
HeartDiseaseorAttack
0.0    0.8968
1.0    0.1032



Asociación numéricas vs target:
 variable  media_sin_evento  media_con_evento  diferencia_std       p_valor
      BMI             28.59             29.48          0.1312 2.290721e-127
 MentHlth              3.37              4.70          0.1734  9.257134e-19
 PhysHlth              4.15              9.22          0.5603  0.000000e+00
  GenHlth              2.51              3.37          0.8099  0.000000e+00
      Age              7.85             10.13          0.7350  0.000000e+00
Education              5.01              4.74         -0.2711 4.318775e-308
   Income              5.98              5.13         -0.4040  0.000000e+00

Asociación binarias vs target:
         variable                 tasa_evento_por_nivel       p_valor
           HighBP            {0.0: 0.0473, 1.0: 0.1703}  0.000000e+00
         HighChol            {0.0: 0.0555, 1.0: 0.1635}  0.000000e+00
        CholCheck            {0.0: 0.0291, 1.0: 0.1063} 9.657383e-127
           Smoker            {0.0: 0.0734, 1.0:


Figuras guardadas en reports/figures/


### Figura 1: distribución de las variables numéricas

Cada panel muestra cuántos encuestados hay en cada valor de una variable, ya sin duplicados.

- **BMI (índice de masa corporal):** concentrado entre 22 y 32, con una cola hacia valores altos. Por eso se vigila el rango de 12 a 98 en la validación.
- **MentHlth y PhysHlth (días con mala salud mental y física en el último mes):** la mayoría de las personas reporta cero días; la distribución está muy sesgada a la derecha.
- **Age, Education e Income:** son códigos ordinales de la encuesta, no edades ni ingresos en dólares. Se distribuyen de forma relativamente uniforme.
- **GenHlth (salud general, 1 a 5):** la mayoría se ubica entre 2 y 3.
- **Smoker:** variable binaria; alrededor de la mitad de los encuestados reporta haber fumado al menos 100 cigarrillos.

Conclusión: las variables continuas tienen colas largas, lo que justifica usar pruebas no paramétricas (Mann-Whitney U) en la fase de asociación.

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(12, 6))
for eje, col in zip(ejes.flat, NUMERICAS + ["Smoker"]):
    sns.histplot(df_sin_dup[col], bins=30, ax=eje, color="#2a7ab0")
    eje.set_title(col)
fig.tight_layout()
fig.savefig(FIGURAS / "01_distribuciones.png", dpi=110)

### Figura 2: matriz de correlación

Muestra la correlación de Pearson entre todas las variables y el objetivo (HeartDiseaseorAttack). Los valores van de -1 a 1; los rojos son positivos y los azules negativos.

- **Correlaciones más altas con el objetivo:** edad, salud general (GenHlth), dificultad para caminar (DiffWalk), presión alta (HighBP), colesterol alto (HighChol) y antecedente de accidente cerebrovascular (Stroke).
- **Correlaciones entre predictoras:** algunas son moderadas, por ejemplo entre presión alta y colesterol alto, o entre salud general y dificultad para caminar. Esto no impide los modelos, pero sí hace que sus coeficientes deban leerse con cuidado.
- **Importante:** una correlación baja no significa que la variable no sirva. Los modelos pueden captar relaciones no lineales que la correlación no mide.

In [ ]:
fig, eje = plt.subplots(figsize=(7.5, 7.5))
sns.heatmap(df_sin_dup[NUMERICAS + BINARIAS + [TARGET]].corr(), cmap="RdBu_r",
            center=0, annot=True, fmt=".1f", annot_kws={"size": 7}, ax=eje)
eje.set_title("Matriz de correlación (sin duplicados)")
fig.tight_layout()
fig.savefig(FIGURAS / "02_correlacion.png", dpi=110)

### Figura 3: tasa de evento por edad y por salud general

Cada barra es la proporción de personas con infarto o enfermedad cardiaca dentro de un grupo.

- **Edad (código BRFSS 1 a 13, grupos de 5 años):** la tasa sube de forma marcada con la edad. Pasa de unos 0,5 % en los grupos más jóvenes a cerca de 25 % en los mayores de 80 años.
- **Salud general (1 = excelente, 5 = mala):** la tasa aumenta en cada escalón. Quienes califican su salud como mala presentan una tasa varias veces mayor que quienes la califican como excelente.

Esta figura muestra por qué la edad y la salud general aparecen primero en la importancia de los modelos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=df_sin_dup.groupby("Age")[TARGET].mean().reset_index(),
            x="Age", y=TARGET, ax=ejes[0], color="#c0392b")
ejes[0].set_title("Tasa de infarto/enfermedad cardiaca por grupo de edad (código BRFSS)")
sns.barplot(data=df_sin_dup.groupby("GenHlth")[TARGET].mean().reset_index(),
            x="GenHlth", y=TARGET, ax=ejes[1], color="#8e44ad")
ejes[1].set_title("Tasa por salud general (1 = excelente, 5 = mala)")
fig.tight_layout()
fig.savefig(FIGURAS / "03_tasas_edad_salud.png", dpi=110)

## Fase 3 (CRISP-DM): preparación de los datos y base de datos SQLite

CRISP-DM fase 3: preparación de los datos y carga en base de datos SQLite.

Crea data/heart_disease.db con dos tablas:
  - raw_encuesta:  datos originales (253.680 filas).
  - encuestados:   datos sin duplicados, validados y con variables derivadas.

In [5]:
import sqlite3
from pathlib import Path

import pandas as pd

CSV = RAIZ / "data" / "heart_disease_health_indicators_BRFSS2015.csv"
BD = RAIZ / "data" / "heart_disease.db"

df_raw = pd.read_csv(CSV)

## Validación de códigos y rangos (según el libro de códigos BRFSS)

In [6]:
RANGOS = {
    "BMI": (12, 98), "MentHlth": (0, 30), "PhysHlth": (0, 30),
    "GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6), "Income": (1, 8),
    "Diabetes": (0, 2),
}
BINARIAS = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "PhysActivity",
            "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
            "DiffWalk", "Sex", "HeartDiseaseorAttack"]

errores = []
for col, (lo, hi) in RANGOS.items():
    fuera = int(((df_raw[col] < lo) | (df_raw[col] > hi)).sum())
    if fuera:
        errores.append(f"{col}: {fuera} valores fuera de [{lo}, {hi}]")
for col in BINARIAS:
    if not set(df_raw[col].dropna().unique()) <= {0, 1}:
        errores.append(f"{col}: valores distintos de 0/1")
print("Validación:", "OK" if not errores else errores)

Validación: OK


## Limpieza

In [7]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Filas originales: {len(df_raw)}  |  sin duplicados: {len(df)}")

Filas originales: 253680  |  sin duplicados: 229781


## Variables derivadas

### Código de edad BRFSS: 1 = 18-24 años, ..., 13 = 80 años o más (grupos de 5 años).

In [8]:
EDAD_APROX = {1: 21, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57,
              9: 62, 10: 67, 11: 72, 12: 77, 13: 82}
df["edad_aprox"] = df["Age"].map(EDAD_APROX)
df["categoria_imc"] = pd.cut(df["BMI"], bins=[0, 18.5, 25, 30, 1000],
                             labels=["bajo_peso", "normal", "sobrepeso", "obesidad"],
                             right=False).astype(str)
df["num_condiciones"] = df[["HighBP", "HighChol", "Diabetes", "Smoker", "Stroke"]].gt(0).sum(axis=1)
df["inactivo"] = (1 - df["PhysActivity"]).astype(int)
df["sexo"] = df["Sex"].map({0: "Mujer", 1: "Hombre"})

## Carga en SQLite

In [9]:
if BD.exists():
    BD.unlink()
with sqlite3.connect(BD) as con:
    df_raw.to_sql("raw_encuesta", con, index=False)
    df.to_sql("encuestados", con, index=False)
    con.execute("CREATE INDEX idx_enc_target ON encuestados(HeartDiseaseorAttack)")
    con.execute("CREATE INDEX idx_enc_edad ON encuestados(Age)")
    con.commit()

with sqlite3.connect(BD) as con:
    n = con.execute("SELECT COUNT(*) FROM encuestados").fetchone()[0]
    tasas = pd.read_sql(
        "SELECT Age AS grupo_edad_brfss, ROUND(AVG(HeartDiseaseorAttack),4) AS tasa_evento, "
        "COUNT(*) AS n FROM encuestados GROUP BY Age ORDER BY Age", con)
print(f"\nBD creada en {BD} con {n} filas en 'encuestados'.")
mostrar(tasas)


BD creada en F:\Esteban\Alejandro Pedreros\heart-disease-crispdm\data\heart_disease.db con 229781 filas en 'encuestados'.
 grupo_edad_brfss  tasa_evento     n
              1.0       0.0053  5512
              2.0       0.0076  7068
              3.0       0.0126 10025
              4.0       0.0158 12234
              5.0       0.0250 14050
              6.0       0.0411 17299
              7.0       0.0615 23140
              8.0       0.0823 27301
              9.0       0.1125 29736
             10.0       0.1418 29168
             11.0       0.1768 22041
             12.0       0.1998 15394
             13.0       0.2455 16813


## Fases 4 y 5 (CRISP-DM): modelamiento y evaluación

# 4. Modelamiento y 5. Evaluación

Lee la tabla `encuestados` desde `data/heart_disease.db`, compara modelos con
validación cruzada y evalúa en un conjunto de prueba (20 %).

Modelos: línea base, **Regresión logística** y **Random Forest** (el mismo
par de familias que usa el repositorio de referencia), más un
**gradient boosting** como comparación adicional.

Explicabilidad: **SHAP** sobre el Random Forest. Diagnóstico: **curvas de
aprendizaje** y **curva de ganancia por deciles**.

In [10]:
import os
import sqlite3
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay,
                             average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                     learning_curve, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BD = RAIZ / "data" / "heart_disease.db"
FIGURAS = RAIZ / "reports" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)
SEMILLA = 42

C:\Users\ESTEBAN\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4.1. Datos y partición

In [11]:
with sqlite3.connect(BD) as con:
    df = pd.read_sql("SELECT * FROM encuestados", con)

TARGET = "HeartDiseaseorAttack"
FEATURES = ["HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke", "Diabetes",
            "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
            "NoDocbcCost", "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex",
            "Age", "Education", "Income"]

X = df[FEATURES]
y = df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
print(f"Entrenamiento: {len(X_tr)}  |  prueba: {len(X_te)}  |  "
      f"prevalencia en prueba: {y_te.mean():.4f}")

Entrenamiento: 183824  |  prueba: 45957  |  prevalencia en prueba: 0.1032


## 4.2. Construcción de los modelos

- **Regresión logística:** requiere escalado (`StandardScaler`) porque penaliza
  por igual todas las variables.
- **Random Forest y gradient boosting:** basados en árboles, no requieren escalado.
- `class_weight="balanced"` compensa el desbalance (10 % de positivos).

In [12]:
modelos = {
    "Línea base (mayoritaria)": make_pipeline(
        StandardScaler(), DummyClassifier(strategy="most_frequent")),
    "Regresión logística": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=50,
        class_weight="balanced_subsample", n_jobs=4, random_state=SEMILLA),
    "Gradient boosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=SEMILLA),
}

## 4.3. Validación cruzada (solo entrenamiento, 5 folds)

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
filas = []
for nombre, modelo in modelos.items():
    r = cross_validate(modelo, X_tr, y_tr, cv=cv,
                       scoring=["roc_auc", "average_precision", "f1"], n_jobs=1)
    filas.append({
        "modelo": nombre,
        "cv_roc_auc": round(r["test_roc_auc"].mean(), 4),
        "cv_roc_auc_std": round(r["test_roc_auc"].std(), 4),
        "cv_pr_auc": round(r["test_average_precision"].mean(), 4),
        "cv_f1": round(r["test_f1"].mean(), 4),
    })
tabla_cv = pd.DataFrame(filas)
print("=== VALIDACIÓN CRUZADA (5 folds) ===")
mostrar(tabla_cv)
tabla_cv.to_csv(RAIZ / "reports" / "metricas_validacion_cruzada.csv", index=False)

C:\Users\ESTEBAN\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ESTEBAN\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


=== VALIDACIÓN CRUZADA (5 folds) ===
                  modelo  cv_roc_auc  cv_roc_auc_std  cv_pr_auc  cv_f1
Línea base (mayoritaria)      0.5000          0.0000     0.1032 0.0000
     Regresión logística      0.8356          0.0025     0.3727 0.3834
           Random Forest      0.8364          0.0029     0.3743 0.3814
       Gradient boosting      0.8375          0.0025     0.3776 0.3785


## 5. Evaluación en el conjunto de prueba

In [14]:
filas = []
probs = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_tr, y_tr)
    prob = modelo.predict_proba(X_te)[:, 1]
    probs[nombre] = prob
    pred = (prob >= 0.5).astype(int)
    filas.append({
        "modelo": nombre,
        "roc_auc": round(roc_auc_score(y_te, prob), 4),
        "pr_auc": round(average_precision_score(y_te, prob), 4),
        "precision_umbral_0.5": round(precision_score(y_te, pred, zero_division=0), 4),
        "recall_umbral_0.5": round(recall_score(y_te, pred, zero_division=0), 4),
        "f1_umbral_0.5": round(f1_score(y_te, pred, zero_division=0), 4),
    })
tabla_test = pd.DataFrame(filas)
print("\n=== EVALUACIÓN EN PRUEBA (20 %) ===")
mostrar(tabla_test)
tabla_test.to_csv(RAIZ / "reports" / "metricas_test.csv", index=False)


=== EVALUACIÓN EN PRUEBA (20 %) ===
                  modelo  roc_auc  pr_auc  precision_umbral_0.5  recall_umbral_0.5  f1_umbral_0.5
Línea base (mayoritaria)   0.5000  0.1032                0.0000             0.0000         0.0000
     Regresión logística   0.8355  0.3725                0.2524             0.7782         0.3811
           Random Forest   0.8369  0.3714                0.2489             0.7944         0.3791
       Gradient boosting   0.8383  0.3767                0.2455             0.8014         0.3759


### Curvas ROC

Matriz de confusión del mejor modelo por ROC-AUC

In [15]:
fig, eje = plt.subplots(figsize=(6, 5))
for nombre, prob in probs.items():
    RocCurveDisplay.from_predictions(y_te, prob, name=nombre, ax=eje)
eje.plot([0, 1], [0, 1], "k--", lw=0.8)
eje.set_title("Curvas ROC en prueba")
fig.tight_layout()
fig.savefig(FIGURAS / "04_curvas_roc.png", dpi=110)


### Figura 4: curvas ROC en el conjunto de prueba

La curva ROC muestra, para cada umbral, la proporción de casos positivos detectados frente a la proporción de falsas alarmas. Cuanto más se acerca a la esquina superior izquierda, mejor. La diagonal punteada es la línea base de un modelo que adivina al azar (AUC 0,50).

- Los tres modelos (regresión logística, Random Forest y gradient boosting) quedan prácticamente superpuestos, con AUC cercano a 0,84.
- La diferencia entre ellos es menor que el intervalo de confianza, por lo que no hay un modelo claramente superior.

In [15]:
fig, eje = plt.subplots(figsize=(6, 5))
for nombre, prob in probs.items():
    RocCurveDisplay.from_predictions(y_te, prob, name=nombre, ax=eje)
eje.plot([0, 1], [0, 1], "k--", lw=0.8)
eje.set_title("Curvas ROC en prueba")
fig.tight_layout()
fig.savefig(FIGURAS / "04_curvas_roc.png", dpi=110)


### Figura 5: matriz de confusión del mejor modelo (umbral 0,5)

Cruza lo que predijo el modelo con lo que realmente ocurrió:

- **Arriba a la izquierda:** personas sin evento que el modelo clasifica como sin evento (verdaderos negativos).
- **Abajo a la derecha:** personas con evento que el modelo detecta (verdaderos positivos).
- **Arriba a la derecha:** falsas alarmas (sin evento clasificadas como con evento).
- **Abajo a la izquierda:** casos reales que el modelo no detecta (falsos negativos).

Con umbral 0,5, el modelo detecta cerca del 80 % de los casos reales, pero una parte importante de las alertas son falsas. Para un tamizaje esto suele ser aceptable; el umbral puede ajustarse según el costo de cada tipo de error.

In [ ]:
mejor = tabla_test[tabla_test["modelo"] != "Línea base (mayoritaria)"] \
    .sort_values("roc_auc", ascending=False).iloc[0]["modelo"]
pred_mejor = (probs[mejor] >= 0.5).astype(int)
fig, eje = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_te, pred_mejor, ax=eje, cmap="Blues", values_format="d",
                                        display_labels=["Sin evento", "Con evento"])
eje.set_title(f"Matriz de confusión (umbral 0,5)\n{mejor}")
fig.tight_layout()
fig.savefig(FIGURAS / "05_matriz_confusion.png", dpi=110)


### 5.1. Ganancia por deciles
Se ordenan los encuestados de prueba por probabilidad predicha y se mide la tasa
real de evento en cada decil. Permite priorizar a quién contactar primero.

In [16]:
decil_df = pd.DataFrame({"prob": probs["Random Forest"], "y": y_te.to_numpy()})
decil_df["decil"] = pd.qcut(decil_df["prob"].rank(method="first", ascending=False),
                            10, labels=range(1, 11))
ganancia = decil_df.groupby("decil", observed=True)["y"].agg(["mean", "size"]) \
    .rename(columns={"mean": "tasa_evento", "size": "n"})
ganancia["lift"] = ganancia["tasa_evento"] / y_te.mean()
ganancia.round(4).to_csv(RAIZ / "reports" / "ganancia_por_decil_random_forest.csv")
print("\nTasa de evento por decil (Random Forest, 1 = mayor riesgo predicho):")
mostrar(ganancia.round(4))

fig, eje = plt.subplots(figsize=(7, 4))
eje.bar(ganancia.index.astype(int), ganancia["tasa_evento"], color="#c0392b")
eje.axhline(y_te.mean(), color="k", ls="--", lw=0.8, label="tasa media")
eje.set_xlabel("Decil de riesgo predicho (1 = mayor)")
eje.set_ylabel("Tasa real de evento")
eje.set_title("Ganancia por deciles (Random Forest, prueba)")
eje.legend()
fig.tight_layout()
fig.savefig(FIGURAS / "06_ganancia_por_decil.png", dpi=110)


Tasa de evento por decil (Random Forest, 1 = mayor riesgo predicho):
       tasa_evento     n    lift
decil                           
1           0.4003  4596  3.8791
2           0.2363  4596  2.2895
3           0.1469  4595  1.4234
4           0.1057  4596  1.0246
5           0.0638  4596  0.6177
6           0.0385  4595  0.3732
7           0.0181  4596  0.1750
8           0.0118  4595  0.1139
9           0.0078  4596  0.0759
10          0.0028  4596  0.0274


## 5.2. Curvas de aprendizaje
Diagnóstico de sobreajuste: si la brecha entre entrenamiento y validación es
grande, el modelo memoriza; si ambas curvas están cerca, generaliza bien.

In [17]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for eje, nombre in zip(ejes, ["Regresión logística", "Random Forest"]):
    tam, tr_sc, va_sc = learning_curve(
        modelos[nombre], X_tr, y_tr, cv=3, scoring="average_precision",
        train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=1, random_state=SEMILLA)
    eje.plot(tam, tr_sc.mean(axis=1), "o-", label="entrenamiento")
    eje.plot(tam, va_sc.mean(axis=1), "o-", label="validación")
    eje.set_title(f"Curva de aprendizaje: {nombre}")
    eje.set_xlabel("Tamaño de entrenamiento")
    eje.set_ylabel("PR-AUC")
    eje.legend()
fig.tight_layout()
fig.savefig(FIGURAS / "07_curvas_aprendizaje.png", dpi=110)

## 5.3. Explicabilidad con SHAP (Random Forest)
Se usa una muestra de 300 registros de prueba: con 300 árboles de profundidad 12,
TreeExplainer tarda demasiado con muestras grandes. check_additivity se desactiva
para acelerar; la suma de SHAP se verificó en una corrida exploratoria.

### En algunas versiones devuelve una lista por clase; se toma la clase positiva.

In [18]:
rf = modelos["Random Forest"]
muestra = X_te.sample(300, random_state=SEMILLA)
explicador = shap.TreeExplainer(rf)
valores = explicador.shap_values(muestra, check_additivity=False)
if isinstance(valores, list):
    valores = valores[1]
elif valores.ndim == 3:
    valores = valores[:, :, 1]

fig = plt.figure()
shap.summary_plot(valores, muestra, show=False, max_display=15)
plt.tight_layout()
plt.savefig(FIGURAS / "08_shap_resumen.png", dpi=110, bbox_inches="tight")

importancia_shap = pd.Series(np.abs(valores).mean(axis=0), index=FEATURES) \
    .sort_values(ascending=False)
importancia_shap.to_csv(RAIZ / "reports" / "importancia_shap_random_forest.csv",
                        header=["abs_shap_medio"])
print("\nImportancia SHAP media absoluta (Random Forest), top 10:")
mostrar(importancia_shap.head(10).round(4))


Importancia SHAP media absoluta (Random Forest), top 10:
Age         0.1069
HighBP      0.0772
GenHlth     0.0712
HighChol    0.0604
Sex         0.0513
DiffWalk    0.0355
Smoker      0.0254
Diabetes    0.0224
Stroke      0.0191
PhysHlth    0.0180


## 5.4. Importancia por permutación (gradient boosting, comparación)

In [19]:
gb = modelos["Gradient boosting"]
perm = permutation_importance(gb, X_te, y_te, scoring="average_precision",
                              n_repeats=3, random_state=SEMILLA, n_jobs=1)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
imp.to_csv(RAIZ / "reports" / "importancia_permutacion_gb.csv", header=["reduccion_pr_auc"])
print("\nImportancia por permutación (gradient boosting), top 10:")
mostrar(imp.head(10).round(4))


Importancia por permutación (gradient boosting), top 10:
Age          0.0793
GenHlth      0.0720
Stroke       0.0395
Sex          0.0320
HighChol     0.0205
HighBP       0.0163
Diabetes     0.0082
Smoker       0.0076
DiffWalk     0.0050
CholCheck    0.0016


## 5.5. Coeficientes de la regresión logística

In [20]:
logit = modelos["Regresión logística"]
coef = pd.Series(logit.named_steps["logisticregression"].coef_[0], index=FEATURES)
coef = coef.reindex(coef.abs().sort_values(ascending=False).index)
coef.to_csv(RAIZ / "reports" / "coeficientes_regresion_logistica.csv", header=["coeficiente"])
print("\nCoeficientes de la regresión logística (estandarizados):")
mostrar(coef.round(4))


Coeficientes de la regresión logística (estandarizados):
Age                  0.8271
GenHlth              0.4994
Sex                  0.4018
HighChol             0.3044
HighBP               0.2690
Stroke               0.2283
Smoker               0.1786
DiffWalk             0.1303
CholCheck            0.1166
Diabetes             0.1067
Income              -0.0936
NoDocbcCost          0.0833
HvyAlcoholConsump   -0.0769
MentHlth             0.0363
PhysActivity         0.0267
Veggies              0.0212
PhysHlth             0.0200
Education            0.0119
Fruits               0.0099
AnyHealthcare       -0.0081
BMI                 -0.0068


## Complemento: robustez, equidad y calibración (evaluación)

# 6. Robustez y equidad (complemento de las fases 4 y 5)

Reajusta la regresión logística y el Random Forest con la misma partición de la
fase 3 y responde tres preguntas:

1. ¿Qué tan estables son las métricas? Intervalos de confianza por bootstrap
   (1.000 remuestreos del conjunto de prueba).
2. ¿El modelo funciona igual para hombres y mujeres, y para mayores y menores de 65 años?
3. ¿Las probabilidades están calibradas? Brier score y tabla de calibración.

In [21]:
import os
import sqlite3
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BD = RAIZ / "data" / "heart_disease.db"
REP = RAIZ / "reports"
FIG = REP / "figures"
SEMILLA = 42
B = 1000

## 6.1. Misma partición que la fase 3

In [22]:
with sqlite3.connect(BD) as con:
    df = pd.read_sql("SELECT * FROM encuestados", con)

TARGET = "HeartDiseaseorAttack"
FEATURES = ["HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke", "Diabetes",
            "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
            "NoDocbcCost", "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex",
            "Age", "Education", "Income"]
X = df[FEATURES]
y = df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
te = X_te.copy()
te["y"] = y_te.values

## 6.2. Reajuste de los modelos principales

In [23]:
logit = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50,
                            class_weight="balanced_subsample", n_jobs=4,
                            random_state=SEMILLA)
logit.fit(X_tr, y_tr)
rf.fit(X_tr, y_tr)
p = {"Regresión logística": logit.predict_proba(X_te)[:, 1],
     "Random Forest": rf.predict_proba(X_te)[:, 1]}
print("Modelos reajustados.")

Modelos reajustados.


## 6.3. Intervalos de confianza por bootstrap

Se remuestrea el conjunto de prueba con reemplazo. El intervalo del 95 % va de
los percentiles 2,5 y 97,5 de las métricas obtenidas.

In [24]:
rng = np.random.default_rng(SEMILLA)
idx = [rng.integers(0, len(y_te), len(y_te)) for _ in range(B)]
yt = y_te.values
filas = []
for nombre, pr in p.items():
    auc_b, pr_b = [], []
    for ii in idx:
        if yt[ii].min() == yt[ii].max():
            continue
        auc_b.append(roc_auc_score(yt[ii], pr[ii]))
        pr_b.append(average_precision_score(yt[ii], pr[ii]))
    for metrica, valor, boot in [("ROC-AUC", roc_auc_score(yt, pr), auc_b),
                                 ("PR-AUC", average_precision_score(yt, pr), pr_b)]:
        lo, hi = np.percentile(boot, [2.5, 97.5])
        filas.append({"modelo": nombre, "metrica": metrica,
                      "valor": round(valor, 4),
                      "ic95_inferior": round(lo, 4),
                      "ic95_superior": round(hi, 4)})
tabla_ic = pd.DataFrame(filas)
mostrar(tabla_ic)
tabla_ic.to_csv(REP / "intervalos_confianza_bootstrap.csv", index=False)

             modelo metrica  valor  ic95_inferior  ic95_superior
Regresión logística ROC-AUC 0.8355         0.8300         0.8407
Regresión logística  PR-AUC 0.3725         0.3597         0.3873
      Random Forest ROC-AUC 0.8369         0.8314         0.8420
      Random Forest  PR-AUC 0.3714         0.3575         0.3857


## 6.4. Equidad: métricas por subgrupo

Se calculan ROC-AUC, recall y precisión (umbral 0,5) para cada grupo. Una brecha
grande entre grupos indica que el modelo no funciona igual para todos.

In [25]:
def metricas_grupo(mascara, pr):
    yy = te.loc[mascara, "y"].values
    pp = pr[mascara.values]
    pred = (pp >= 0.5).astype(int)
    tp = int(((pred == 1) & (yy == 1)).sum())
    fp = int(((pred == 1) & (yy == 0)).sum())
    fn = int(((pred == 0) & (yy == 1)).sum())
    return {
        "n": len(yy),
        "prevalencia": round(yy.mean(), 4),
        "roc_auc": round(roc_auc_score(yy, pp), 4) if len(np.unique(yy)) > 1 else np.nan,
        "recall": round(tp / (tp + fn), 4) if (tp + fn) else np.nan,
        "precision": round(tp / (tp + fp), 4) if (tp + fp) else np.nan,
    }

grupos = {
    "Sexo: hombre (1)": te["Sex"] == 1,
    "Sexo: mujer (0)": te["Sex"] == 0,
    "Edad: 65 años o más (código 10 a 13)": te["Age"] >= 10,
    "Edad: menores de 65 (código 1 a 9)": te["Age"] < 10,
}
filas = []
for nombre, pr in p.items():
    for gnombre, m in grupos.items():
        filas.append({"modelo": nombre, "grupo": gnombre, **metricas_grupo(m, pr)})
tabla_eq = pd.DataFrame(filas)
mostrar(tabla_eq)
tabla_eq.to_csv(REP / "metricas_por_subgrupo.csv", index=False)

             modelo                                grupo     n  prevalencia  roc_auc  recall  precision
Regresión logística                     Sexo: hombre (1) 20239       0.1336   0.8220  0.8321     0.2680
Regresión logística                      Sexo: mujer (0) 25718       0.0793   0.8366  0.7067     0.2313
Regresión logística Edad: 65 años o más (código 10 a 13) 16620       0.1841   0.7552  0.8719     0.2649
Regresión logística   Edad: menores de 65 (código 1 a 9) 29337       0.0574   0.8479  0.6081     0.2247
      Random Forest                     Sexo: hombre (1) 20239       0.1336   0.8238  0.8421     0.2695
      Random Forest                      Sexo: mujer (0) 25718       0.0793   0.8376  0.7312     0.2229
      Random Forest Edad: 65 años o más (código 10 a 13) 16620       0.1841   0.7574  0.8807     0.2638
      Random Forest   Edad: menores de 65 (código 1 a 9) 29337       0.0574   0.8499  0.6378     0.2180


## 6.5. Calibración

Brier score (menor es mejor) y tabla de calibración por deciles de probabilidad
predicha. Si la tasa observada se parece a la predicha, las probabilidades son
confiables para usarlas como riesgo.

In [26]:
filas = []
for nombre, pr in p.items():
    filas.append({"modelo": nombre,
                  "brier": round(brier_score_loss(yt, pr), 4),
                  "prevalencia_media_predicha": round(pr.mean(), 4),
                  "prevalencia_observada": round(yt.mean(), 4)})
tabla_brier = pd.DataFrame(filas)
mostrar(tabla_brier)
tabla_brier.to_csv(REP / "calibracion_brier.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 5))
for nombre, pr in p.items():
    deciles = pd.qcut(pr, 10, labels=False, duplicates="drop")
    d = pd.DataFrame({"pred": pr, "obs": yt, "dec": deciles}).groupby("dec").mean()
    ax.plot(d["pred"], d["obs"], marker="o", label=nombre)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Calibración perfecta")
ax.set_xlabel("Probabilidad predicha (media por decil)")
ax.set_ylabel("Tasa observada")
ax.set_title("Curva de calibración en prueba")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "09_calibracion.png", dpi=130)
print("Figura guardada: 09_calibracion.png")

             modelo  brier  prevalencia_media_predicha  prevalencia_observada
Regresión logística 0.1709                      0.3643                 0.1032
      Random Forest 0.1654                      0.3623                 0.1032


Figura guardada: 09_calibracion.png
